# Leaf classifier for Field Assistant (coffee · beans · maize)

Trains a **small on-device image classifier** (MobileNetV3-Large, ~6 MB TFLite) that tells Noor's three crops'
common leaf problems apart, plus an **`other`** class for anything else. The Flutter app runs it offline and only
lets the LLM explain a result when the classifier is confident; otherwise it says *"not sure — ask a person"*.

**Run on:** Google Colab (Runtime → Change runtime type → **T4 GPU**) or Kaggle (GPU on, Internet on).
Full run ≈ 30–45 min, most of it downloading data.

**Outputs** (in `export/`, zipped at the end):
| file | goes to |
|---|---|
| `leaf_classifier.tflite` | `field_assistant/assets/models/` |
| `leaf_classifier.json` (labels, temperature, threshold) | `field_assistant/assets/models/` |
| `MODEL_CARD.md`, `confusion_test.png` | pitch / submission (data sources, licences, gaps, per-source accuracy) |

**What the data does NOT cover (scored in judging, so it is written into the model card automatically):**
studio/plain-background images (PlantVillage, BRACOL) inflate accuracy; only *leaves* — no coffee berry disease,
berry borer, wilt, nutrient deficiency, drought; no images from Noor's region; healthy maize only from studio images.

In [ ]:
# ---------------------------------------------------------------- configuration
import os
SMOKE = os.environ.get("SMOKE") == "1"   # tiny CPU run to test the pipeline end to end

IMG = 224                 # model input (must match the app: lib/leaf_classifier.dart reads it from the json)
STORE = 256               # images are stored with shorter side = STORE, then center-cropped to a square
MAX_PER_CLASS = 30 if SMOKE else 1200      # cap per (source, label) for TRAIN; keeps classes balanced and runs fast
OTHER_PER_PV_CLASS = 3 if SMOKE else 40    # PlantVillage non-target crops sampled into `other`
IMAGENETTE_OTHER = 0 if SMOKE else 600     # non-plant photos for `other` (set 0 to skip)
EPOCHS_HEAD, EPOCHS_FT = (1, 1) if SMOKE else (6, 8)
BATCH = 16 if SMOKE else 64
TARGET_PRECISION = 0.90   # threshold is the lowest confidence at which accepted answers are >= 90% right
MIN_THRESHOLD = 0.50
SEED = 42

# Which sources to use. All are public; the notebook skips a source (with a warning) if it cannot download it.
USE = dict(
    plantvillage=True,  # GitHub, studio images (maize + `other` crops)
    plantdoc=True,      # GitHub, field/web images (maize + `other`)
    bracol=True,        # GitHub, Brazil arabica coffee, white background — has a clean train/val/test split
    jmuben=not SMOKE,   # Hugging Face, Kenya arabica coffee (pre-augmented: used for TRAINING only)
    beans=not SMOKE,    # Hugging Face, Uganda iBean field photos (Makerere AI Lab)
    imagenette=IMAGENETTE_OTHER > 0,  # fast.ai, generic photos → `other`
)
# Sources trusted for calibration (temperature + threshold): field photos or clean splits, not studio/augmented.
CALIB_SOURCES = ["beans", "plantdoc", "bracol", "imagenette"]

WORK = "/content" if os.path.isdir("/content") else os.path.abspath("leaf_work")
RAW, DATA, EXPORT = f"{WORK}/raw", f"{WORK}/data", f"{WORK}/export"
for d in (RAW, DATA, EXPORT):
    os.makedirs(d, exist_ok=True)
print("SMOKE" if SMOKE else "FULL", "run; working dir:", WORK)

In [ ]:
# ---------------------------------------------------------------- imports
import io, json, random, shutil, subprocess, tarfile, urllib.request, zipfile
from collections import Counter
import numpy as np, pandas as pd
from PIL import Image, ImageOps
import tensorflow as tf, keras
print("TF", tf.__version__, "| Keras", keras.__version__, "| GPU:", tf.config.list_physical_devices("GPU"))
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)
try:
    import datasets  # Hugging Face; preinstalled on Colab/Kaggle
except ImportError:
    subprocess.run("pip install -q datasets", shell=True, check=False)
    try:
        import datasets
    except ImportError:
        datasets = None

## Label scheme
One flat list of `crop__condition` labels. Each source's own class names are mapped onto it.
Add a crop later by adding labels here and a mapping in its loader.

In [ ]:
LABELS = [
    # id                            crop     condition (display, English — the LLM translates)
    ("coffee__healthy",            "Coffee", "healthy leaf"),
    ("coffee__leaf_rust",          "Coffee", "leaf rust"),
    ("coffee__cercospora",         "Coffee", "Cercospora leaf spot (brown eye spot)"),
    ("coffee__phoma",              "Coffee", "Phoma leaf spot"),
    ("coffee__leaf_miner",         "Coffee", "leaf miner (insect damage)"),
    ("bean__healthy",              "Bean",   "healthy leaf"),
    ("bean__angular_leaf_spot",    "Bean",   "angular leaf spot"),
    ("bean__rust",                 "Bean",   "bean rust"),
    ("maize__healthy",             "Maize",  "healthy leaf"),
    ("maize__common_rust",         "Maize",  "common rust"),
    ("maize__gray_leaf_spot",      "Maize",  "gray leaf spot"),
    ("maize__northern_leaf_blight","Maize",  "northern leaf blight"),
    ("other",                      "",       "not one of the crops this model knows"),
]
LABEL_IDS = [l[0] for l in LABELS]
IDX = {l: i for i, l in enumerate(LABEL_IDS)}
OTHER = "other"

In [ ]:
# ---------------------------------------------------------------- helpers
ROWS = []   # manifest: path, label, source, split

def save(img, label, source, split, key):
    # Normalise orientation/colour, shrink so the shorter side is STORE px, write JPEG, record in manifest.
    try:
        if not isinstance(img, Image.Image):
            img = Image.open(img)
        img = ImageOps.exif_transpose(img).convert("RGB")
        w, h = img.size
        s = STORE / min(w, h)
        if s < 1:
            img = img.resize((max(1, round(w * s)), max(1, round(h * s))), Image.BILINEAR)
        out = f"{DATA}/{split}/{label}/{source}_{key}.jpg"
        os.makedirs(os.path.dirname(out), exist_ok=True)
        img.save(out, quality=90)
        ROWS.append(dict(path=out, label=label, source=source, split=split))
    except Exception as e:
        print(f"  skip {source}/{key}: {e}")

def split_of(i, n, val=0.15, test=0.15):
    r = i / max(n, 1)
    return "test" if r < test else "val" if r < test + val else "train"

def cap(items, n):
    items = list(items); random.shuffle(items); return items[:n]

def sh(cmd, cwd=None):
    subprocess.run(cmd, shell=True, check=True, cwd=cwd)

def git_sparse(url, name, paths):
    # Partial clone + checkout of only the listed files: downloads just those images.
    d = f"{RAW}/{name}"
    if not os.path.isdir(d):
        sh(f"git -c gc.auto=0 clone -q --depth 1 --filter=blob:none --no-checkout {url} {d}")
    spec = f"{RAW}/{name}.paths"
    with open(spec, "w") as f:
        f.write("\n".join(paths) + "\n")
    sh(f"git -c gc.auto=0 --literal-pathspecs checkout -q HEAD --pathspec-from-file={spec}", cwd=d)
    return d

def git_ls(d_or_url, name):
    d = f"{RAW}/{name}"
    if not os.path.isdir(d):
        sh(f"git -c gc.auto=0 clone -q --depth 1 --filter=blob:none --no-checkout {d_or_url} {d}")
    out = subprocess.run("git ls-tree -r --name-only HEAD", shell=True, cwd=d, capture_output=True, text=True, check=True)
    return d, out.stdout.splitlines()

def hf_columns(ds):
    feats = ds.features
    img = next(k for k, v in feats.items() if isinstance(v, datasets.Image))
    lab = next(k for k, v in feats.items() if isinstance(v, datasets.ClassLabel))
    return img, lab, feats[lab].names

def loader(name):
    def wrap(fn):
        def run():
            if not USE.get(name):
                print(f"[{name}] disabled"); return
            before = len(ROWS)
            try:
                fn()
            except Exception as e:
                print(f"[{name}] FAILED, skipping this source: {type(e).__name__}: {e}")
            print(f"[{name}] {len(ROWS) - before} images")
        LOADERS.append(run)
        return fn
    return wrap
LOADERS = []

## Data sources
| source | crop(s) | conditions | licence | where it was photographed |
|---|---|---|---|---|
| [PlantVillage](https://github.com/spMohanty/PlantVillage-Dataset) (Hughes & Salathé 2015) | maize (+ other crops → `other`) | studio, plain background | see repo (check terms) | lab, USA |
| [PlantDoc](https://github.com/pratikkayal/PlantDoc-Dataset) (Singh et al. 2020) | maize (+ other crops → `other`) | web/field, cluttered | CC BY 4.0 | various (web-scraped) |
| [BRACOL](https://github.com/esgario/lara2018) (Esgario et al. 2020; Mendeley 10.17632/yy2k5y8mxg.1) | coffee (arabica) | smartphone, white background | CC BY 4.0 (data) / MIT (repo) | Brazil |
| [JMuBEN](https://huggingface.co/datasets/Project-AgML/arabica_coffee_leaf_disease_classification) (Jepkoech et al. 2021) | coffee (arabica) | pre-cropped & augmented | CC BY 4.0 | Kenya |
| [iBean](https://huggingface.co/datasets/AI-Lab-Makerere/beans) (Makerere AI Lab) | beans | smartphone, in the field | MIT | Uganda |
| [Imagenette](https://github.com/fastai/imagenette) | — (→ `other`) | everyday photos | Apache 2.0 (ImageNet subset — check terms) | various |

In [ ]:
@loader("plantvillage")
def load_plantvillage():
    url = "https://github.com/spMohanty/PlantVillage-Dataset.git"
    d, files = git_ls(url, "plantvillage")
    files = [f for f in files if f.startswith("raw/color/") and f.lower().endswith((".jpg", ".jpeg", ".png"))]
    by_cls = {}
    for f in files:
        by_cls.setdefault(f.split("/")[2], []).append(f)
    maize = {
        "Corn_(maize)___healthy": "maize__healthy",
        "Corn_(maize)___Common_rust_": "maize__common_rust",
        "Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot": "maize__gray_leaf_spot",
        "Corn_(maize)___Northern_Leaf_Blight": "maize__northern_leaf_blight",
    }
    picks = []  # (file, label, split)
    for cls, fs in by_cls.items():
        fs = sorted(fs); random.Random(SEED).shuffle(fs)
        if cls in maize:
            n = min(len(fs), int(MAX_PER_CLASS / 0.7))
            picks += [(f, maize[cls], split_of(i, n, 0.1, 0.1)) for i, f in enumerate(fs[:n])]
        else:
            n = OTHER_PER_PV_CLASS
            picks += [(f, OTHER, split_of(i, n, 0.1, 0.1)) for i, f in enumerate(fs[:n])]
    git_sparse(url, "plantvillage", [p[0] for p in picks])
    for i, (f, lab, sp) in enumerate(picks):
        save(f"{d}/{f}", lab, "plantvillage", sp, i)

@loader("plantdoc")
def load_plantdoc():
    url = "https://github.com/pratikkayal/PlantDoc-Dataset.git"
    d, files = git_ls(url, "plantdoc")
    maize = {"Corn rust leaf": "maize__common_rust", "Corn Gray leaf spot": "maize__gray_leaf_spot",
             "Corn leaf blight": "maize__northern_leaf_blight"}
    picks = []
    for f in files:
        parts = f.split("/")
        if len(parts) != 3 or parts[0] not in ("train", "test") or not parts[2].lower().endswith((".jpg", ".jpeg", ".png")):
            continue
        picks.append((f, maize.get(parts[1], OTHER), parts[0]))
    random.Random(SEED).shuffle(picks)
    if SMOKE:
        picks = [p for p in picks if p[1] != OTHER][:60] + [p for p in picks if p[1] == OTHER][:30]
    git_sparse(url, "plantdoc", [p[0] for p in picks])
    seen = Counter()
    for i, (f, lab, sp) in enumerate(picks):
        if sp == "train":  # carve a validation set out of PlantDoc train (official test stays test)
            seen[lab] += 1
            sp = "val" if seen[lab] % 6 == 0 else "train"
        save(f"{d}/{f}", lab, "plantdoc", sp, i)

@loader("bracol")
def load_bracol():
    url = "https://github.com/esgario/lara2018.git"
    d, files = git_ls(url, "bracol")
    csv = "classification/dataset/dataset.csv"
    git_sparse(url, "bracol", [csv])
    df = pd.read_csv(f"{d}/{csv}")
    # predominant_stress codes (lara2018/classification/results.py): 0 healthy, 1 leaf miner, 2 rust, 3 phoma, 4 cercospora
    code2 = {0: "coffee__healthy", 1: "coffee__leaf_miner", 2: "coffee__leaf_rust", 3: "coffee__phoma", 4: "coffee__cercospora"}
    df = df.sample(frac=1, random_state=SEED)
    if SMOKE:
        df = df.groupby("predominant_stress").head(12)
    paths = [f"classification/dataset/leaf/{i}.jpg" for i in df["id"]]
    git_sparse(url, "bracol", paths)
    counts = df["predominant_stress"].value_counts().to_dict(); seen = Counter()
    for (_, r), p in zip(df.iterrows(), paths):
        c = int(r["predominant_stress"]); seen[c] += 1
        save(f"{d}/{p}", code2[c], "bracol", split_of(seen[c] - 1, counts[c]), int(r["id"]))

@loader("jmuben")
def load_jmuben():
    ds = datasets.load_dataset("Project-AgML/arabica_coffee_leaf_disease_classification", split="train")
    img_col, lab_col, names = hf_columns(ds)
    print("  JMuBEN classes:", names)
    m = {"cerscospora": "coffee__cercospora", "cercospora": "coffee__cercospora", "healthy": "coffee__healthy",
         "leaf_rust": "coffee__leaf_rust", "rust": "coffee__leaf_rust", "miner": "coffee__leaf_miner", "phoma": "coffee__phoma"}
    labels = np.array(list(ds[lab_col]))
    for k, name in enumerate(names):
        lab = m.get(name.lower())
        if lab is None:
            print("  unmapped JMuBEN class:", name); continue
        idx = cap(np.where(labels == k)[0].tolist(), MAX_PER_CLASS)
        # JMuBEN ships augmented copies of the same leaves, so a random test split would leak.
        # It is used for TRAINING only; coffee is evaluated on BRACOL.
        for i in idx:
            save(ds[int(i)][img_col], lab, "jmuben", "train", int(i))

@loader("beans")
def load_beans():
    ds = None
    for name in ("AI-Lab-Makerere/beans", "beans"):
        try:
            ds = datasets.load_dataset(name); break
        except Exception as e:
            print("  ", name, "->", e)
    split_map = {"train": "train", "validation": "val", "test": "test"}
    m = {"angular_leaf_spot": "bean__angular_leaf_spot", "bean_rust": "bean__rust", "healthy": "bean__healthy"}
    for hf_split, sp in split_map.items():
        part = ds[hf_split]
        img_col, lab_col, names = hf_columns(part)
        for i in range(len(part)):
            r = part[i]
            save(r[img_col], m[names[r[lab_col]]], "beans", sp, f"{hf_split}{i}")

@loader("imagenette")
def load_imagenette():
    tgz = f"{RAW}/imagenette2-160.tgz"
    if not os.path.exists(tgz):
        urllib.request.urlretrieve("https://s3.amazonaws.com/fast-ai-imageclas/imagenette2-160.tgz", tgz)
    with tarfile.open(tgz) as t:
        members = [m for m in t.getmembers() if m.isfile() and m.name.endswith(".JPEG")]
        for split in ("train", "val"):
            ms = cap([m for m in members if f"/{split}/" in m.name], IMAGENETTE_OTHER if split == "train" else IMAGENETTE_OTHER // 4)
            for i, m in enumerate(ms):
                sp = "train" if split == "train" else ("val" if i % 2 else "test")
                save(Image.open(io.BytesIO(t.extractfile(m).read())), OTHER, "imagenette", sp, i)

In [ ]:
# ---------------------------------------------------------------- build the manifest
shutil.rmtree(DATA, ignore_errors=True); os.makedirs(DATA)
ROWS.clear()
for run in LOADERS:
    run()
man = pd.DataFrame(ROWS)
# cap TRAIN per (source, label) so no single source/class dominates
man = pd.concat([
    g.sample(min(len(g), MAX_PER_CLASS), random_state=SEED) if s == "train" else g
    for (s, _, _), g in man.groupby(["split", "source", "label"])
]).reset_index(drop=True)
man.to_csv(f"{EXPORT}/manifest.csv", index=False)
counts = man.pivot_table(index="label", columns=["split", "source"], aggfunc="size", fill_value=0)
try:
    display(counts)
except NameError:
    print(counts.to_string())
missing = [l for l in LABEL_IDS if l not in set(man[man.split == "train"].label)]
if missing:
    print("WARNING — no training images for:", missing, "(those labels will never be predicted)")

## Input pipeline
Same preprocessing as the app: **center-crop to a square, resize to 224, RGB 0–255 floats**
(MobileNetV3 in Keras contains its own normalisation layer). Training adds flips, rotation, zoom,
brightness/contrast jitter to push the model toward phone-photo conditions.

In [ ]:
def load_img(path):
    x = tf.io.decode_jpeg(tf.io.read_file(path), channels=3)
    h, w = tf.shape(x)[0], tf.shape(x)[1]
    s = tf.minimum(h, w)
    x = tf.image.crop_to_bounding_box(x, (h - s) // 2, (w - s) // 2, s, s)
    return tf.image.resize(x, (IMG, IMG), antialias=True)   # float32 0..255

augment = keras.Sequential([
    keras.layers.RandomFlip("horizontal_and_vertical"),
    keras.layers.RandomRotation(0.25, fill_mode="reflect"),
    keras.layers.RandomZoom((-0.25, 0.1), fill_mode="reflect"),
    keras.layers.RandomBrightness(0.25, value_range=(0, 255)),
    keras.layers.RandomContrast(0.25),
], name="augment")

def make_ds(df, train=False):
    y = df.label.map(IDX).values.astype("int32")
    ds = tf.data.Dataset.from_tensor_slices((df.path.values, y))
    if train:
        ds = ds.shuffle(len(df), seed=SEED, reshuffle_each_iteration=True)
    ds = ds.map(lambda p, l: (load_img(p), l), num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.batch(BATCH)
    if train:
        ds = ds.map(lambda x, l: (tf.clip_by_value(augment(x, training=True), 0, 255), l), num_parallel_calls=tf.data.AUTOTUNE)
    return ds.prefetch(tf.data.AUTOTUNE)

tr, va, te = (man[man.split == s].reset_index(drop=True) for s in ("train", "val", "test"))
ds_tr, ds_va = make_ds(tr, train=True), make_ds(va)
cnt = tr.label.value_counts()
class_weight = {i: 1.0 for i in range(len(LABEL_IDS))}
class_weight.update({IDX[l]: float(len(tr) / (len(cnt) * c)) for l, c in cnt.items()})
print(f"train {len(tr)} · val {len(va)} · test {len(te)}")

## Model: MobileNetV3-Large (ImageNet) → dropout → linear layer (logits)

In [ ]:
def build(weights="imagenet"):
    base = keras.applications.MobileNetV3Large(
        input_shape=(IMG, IMG, 3), include_top=False, weights=weights, pooling="avg", include_preprocessing=True)
    inp = keras.Input((IMG, IMG, 3), name="image")
    x = base(inp, training=False)          # BatchNorm stays in inference mode, also while fine-tuning
    x = keras.layers.Dropout(0.3)(x)
    out = keras.layers.Dense(len(LABEL_IDS), name="logits")(x)   # raw logits; softmax(logits / T) in the app
    return keras.Model(inp, out), base

try:
    model, base = build("imagenet")
except Exception as e:   # e.g. no internet to fetch weights (only expected in the offline smoke test)
    print("ImageNet weights unavailable, training from scratch:", e)
    model, base = build(None)

loss = keras.losses.SparseCategoricalCrossentropy(from_logits=True)
cbs = [keras.callbacks.EarlyStopping(monitor="val_accuracy", patience=3, restore_best_weights=True)]

# stage 1: train the new head only
base.trainable = False
model.compile(keras.optimizers.Adam(1e-3), loss, metrics=["accuracy"])
model.fit(ds_tr, validation_data=ds_va, epochs=EPOCHS_HEAD, class_weight=class_weight, callbacks=cbs)

# stage 2: fine-tune the top of the backbone
base.trainable = True
for layer in base.layers[:-60]:
    layer.trainable = False
for layer in base.layers:
    if isinstance(layer, keras.layers.BatchNormalization):
        layer.trainable = False
model.compile(keras.optimizers.Adam(1e-4), loss, metrics=["accuracy"])
model.fit(ds_tr, validation_data=ds_va, epochs=EPOCHS_FT, class_weight=class_weight, callbacks=cbs)

## Calibration and the "not sure" threshold
1. **Temperature scaling**: one number `T` so that `softmax(logits / T)` is an honest probability
   (fitted on validation photos from field/clean sources only).
2. **Threshold**: lowest confidence at which predictions above it are ≥ 90 % correct on those photos.
   Below it, or when the top label is `other`, the app does not name a disease and points to the extension officer.

In [ ]:
def logits_of(df):
    return model.predict(make_ds(df), verbose=0) if len(df) else np.zeros((0, len(LABEL_IDS)), "float32")

def softmax(z):
    z = z - z.max(1, keepdims=True); e = np.exp(z); return e / e.sum(1, keepdims=True)

cal = va[va.source.isin(CALIB_SOURCES)].reset_index(drop=True)
if len(cal) < 30:
    print("few calibration images from field sources; using the whole validation set")
    cal = va
z_cal, y_cal = logits_of(cal), cal.label.map(IDX).values

def nll(T):
    p = softmax(z_cal / T); return -np.mean(np.log(p[np.arange(len(y_cal)), y_cal] + 1e-12))
Ts = np.linspace(0.5, 5.0, 91)
T = float(Ts[np.argmin([nll(t) for t in Ts])])

def selective(z, y, thr):
    p = softmax(z / T); conf, pred = p.max(1), p.argmax(1)
    accept = (conf >= thr) & (pred != IDX[OTHER])
    acc_all = float((pred == y).mean()) if len(y) else float("nan")
    cov = float(accept.mean()) if len(y) else float("nan")
    sel = float((pred[accept] == y[accept]).mean()) if accept.any() else float("nan")
    return acc_all, sel, cov

thr = None
for t in np.linspace(MIN_THRESHOLD, 0.99, 50):
    _, sel, cov = selective(z_cal, y_cal, t)
    if not np.isnan(sel) and sel >= TARGET_PRECISION and cov > 0:
        thr = float(t); break
if thr is None:
    thr = 0.90
    print(f"WARNING: no threshold reaches {TARGET_PRECISION:.0%} precision on calibration data — using {thr} "
          "(the app will say 'not sure' a lot; more/better field data needed)")
print(f"temperature T = {T:.2f} · threshold = {thr:.2f}")

In [ ]:
# ---------------------------------------------------------------- honest test report, per source
z_te, y_te = logits_of(te), te.label.map(IDX).values
rows = []
for src in ["ALL"] + sorted(te.source.unique()):
    m = np.ones(len(te), bool) if src == "ALL" else (te.source == src).values
    if m.sum() == 0:
        continue
    acc, sel, cov = selective(z_te[m], y_te[m], thr)
    rows.append(dict(source=src, images=int(m.sum()), top1_accuracy=round(acc, 3),
                     accuracy_when_confident=round(sel, 3), share_answered=round(cov, 3)))
report = pd.DataFrame(rows)
print(report.to_string(index=False))

import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix
pred_te = softmax(z_te / T).argmax(1) if len(te) else np.array([], int)
cm = confusion_matrix(y_te, pred_te, labels=range(len(LABEL_IDS)))
fig, ax = plt.subplots(figsize=(9, 8))
ax.imshow(cm, cmap="Greens")
ax.set_xticks(range(len(LABEL_IDS)), LABEL_IDS, rotation=90, fontsize=8); ax.set_yticks(range(len(LABEL_IDS)), LABEL_IDS, fontsize=8)
for i in range(len(LABEL_IDS)):
    for j in range(len(LABEL_IDS)):
        if cm[i, j]:
            ax.text(j, i, cm[i, j], ha="center", va="center", fontsize=7)
ax.set_xlabel("predicted"); ax.set_ylabel("true"); ax.set_title("Test set (all sources)")
fig.tight_layout(); fig.savefig(f"{EXPORT}/confusion_test.png", dpi=130)

## Export to TFLite (float16 weights, float input/output), check runtime compatibility and that it matches Keras

In [ ]:
sm = f"{WORK}/saved_model"
shutil.rmtree(sm, ignore_errors=True)
try:
    model.export(sm, format="tf_saved_model", verbose=False)
except TypeError:  # older Keras without `verbose`
    model.export(sm, format="tf_saved_model")
conv = tf.lite.TFLiteConverter.from_saved_model(sm)
# float16 weights: ~6 MB and loadable by old runtimes. Do NOT use plain Optimize.DEFAULT (int8 "dynamic range"):
# it emits FULLY_CONNECTED v12 and needs TFLite >= 2.17, but tflite_flutter 0.12.1 ships TFLite 2.12 on iOS
# ("Unable to create interpreter").
conv.optimizations = [tf.lite.Optimize.DEFAULT]
conv.target_spec.supported_types = [tf.float16]
tfl = conv.convert()
tfl_path = f"{EXPORT}/leaf_classifier.tflite"
open(tfl_path, "wb").write(tfl)
print(f"leaf_classifier.tflite: {len(tfl) / 1e6:.1f} MB")

def min_runtime(buf):
    # read the "min_runtime_version" metadata the converter writes into the flatbuffer
    from tensorflow.lite.tools import flatbuffer_utils as fu
    m = fu.convert_bytearray_to_object(bytearray(buf))
    for md in m.metadata or []:
        if md.name in (b"min_runtime_version", "min_runtime_version"):
            return bytes(m.buffers[md.buffer].data).rstrip(b"\0").decode()
    return "unknown"
rt = min_runtime(tfl)
print("min TFLite runtime:", rt)
assert rt == "unknown" or tuple(int(v) for v in rt.split(".")[:2]) <= (2, 12), \
    f"model needs TFLite {rt}, the app's iOS runtime is 2.12 — change the converter settings"

interp = tf.lite.Interpreter(model_path=tfl_path)
interp.allocate_tensors()
inp, out = interp.get_input_details()[0], interp.get_output_details()[0]
print("input", inp["shape"], inp["dtype"], "| output", out["shape"], out["dtype"])
assert list(inp["shape"]) == [1, IMG, IMG, 3] and list(out["shape"]) == [1, len(LABEL_IDS)]

# Parity on test images: PIL preprocessing written the way the Dart code does it
# (EXIF orientation → center square crop → resize 224 → RGB floats 0..255).
def app_like(path):
    im = ImageOps.exif_transpose(Image.open(path)).convert("RGB")
    w, h = im.size; s = min(w, h)
    im = im.crop(((w - s) // 2, (h - s) // 2, (w - s) // 2 + s, (h - s) // 2 + s)).resize((IMG, IMG), Image.BOX)
    return np.asarray(im, "float32")[None]

sample = te.sample(min(len(te), 50), random_state=SEED)
agree = 0
for p in sample.path:
    interp.set_tensor(inp["index"], app_like(p)); interp.invoke()
    agree += int(interp.get_tensor(out["index"])[0].argmax() == model.predict(load_img(p)[None], verbose=0)[0].argmax())
print(f"TFLite (app-style preprocessing) vs Keras top-1 agreement: {agree}/{len(sample)}")

In [ ]:
# ---------------------------------------------------------------- app metadata + model card
meta = dict(
    version=1, input_size=IMG, input="RGB float32 0..255, center square crop", output="logits",
    temperature=round(T, 4), threshold=round(thr, 4), other_label=OTHER,
    labels=[dict(id=i, crop=c, condition=d) for i, c, d in LABELS],
    trained_on=sorted(man.source.unique().tolist()),
    # NaN (e.g. a source where nothing was answered) is not valid JSON for Dart -> write null
    test_report=[{k: (None if isinstance(v, float) and np.isnan(v) else v) for k, v in r.items()}
                 for r in report.to_dict(orient="records")],
)
json.dump(meta, open(f"{EXPORT}/leaf_classifier.json", "w"), indent=2, allow_nan=False)  # strict JSON for the app

def as_md(df, **kw):
    try:
        return df.to_markdown(**kw)
    except ImportError:  # tabulate missing
        return '```\n' + df.to_string(**kw) + '\n```'

data_rows = man.groupby(["source", "split"]).size().unstack(fill_value=0)
card = f'''# Leaf classifier — model card

**Model:** MobileNetV3-Large (ImageNet pre-trained), fine-tuned; TFLite with float16 weights, {len(tfl) / 1e6:.1f} MB.
Runs offline on the phone in the Field Assistant app (`tflite_flutter`). Input {IMG}×{IMG} RGB; output {len(LABEL_IDS)} logits.

**Labels:** {", ".join(LABEL_IDS)}

**Fail-safe:** probabilities are temperature-scaled (T = {T:.2f}). The app names a condition only if the top
probability ≥ **{thr:.2f}** and the label is not `other`. That threshold is the lowest at which ≥ {TARGET_PRECISION:.0%} of
accepted answers were correct on validation photos from {", ".join(CALIB_SOURCES)}. Otherwise: "not sure — ask your
extension officer", with an option to save the photo and send it for human review when there is signal.

## Data used (images per split)
{as_md(data_rows)}

Sources: PlantVillage (github.com/spMohanty/PlantVillage-Dataset), PlantDoc (CC BY 4.0), BRACOL (CC BY 4.0,
Mendeley 10.17632/yy2k5y8mxg.1, via github.com/esgario/lara2018), JMuBEN (CC BY 4.0, Jepkoech et al. 2021,
via Hugging Face Project-AgML), iBean (MIT, Makerere AI Lab), Imagenette (fast.ai). Check each licence before reuse.

## Test results (held-out images; "answered" = confident and not `other`)
{as_md(report, index=False)}

Coffee is tested on BRACOL only (Brazil, white background). JMuBEN contains augmented copies, so it is training-only.

## What the data does NOT cover
- **Studio vs. field:** PlantVillage and BRACOL are plain-background images; scores on them overstate field accuracy.
  PlantDoc and iBean are the closest to real phone photos — look at those rows.
- **Only leaves:** no coffee berry disease, coffee berry borer, coffee wilt, nutrient deficiencies, drought or frost
  damage, no stems/roots/fruit. Many causes of falling yield cannot be seen on a leaf at all.
- **Only three crops, a few conditions each.** Anything else should land in `other` → "not sure".
- **Geography:** Brazil, Kenya, Uganda, USA, web images — none from the target highlands; leaf varieties, light and
  camera quality differ.
- **Healthy maize** comes only from studio images.
- **Mixed infections** (two diseases on one leaf) are labelled with one predominant condition.
- Next step: photos reviewed by extension officers via the app's outbox become a local, labelled field set.
'''
open(f"{EXPORT}/MODEL_CARD.md", "w").write(card)

zip_path = shutil.make_archive(f"{WORK}/leaf_classifier_export", "zip", EXPORT)
print("wrote", zip_path, "→", os.listdir(EXPORT))
try:
    from google.colab import files
    files.download(zip_path)
except Exception:
    pass  # Kaggle: download from the Output panel

## Put it in the app
Unzip and copy **`leaf_classifier.tflite`** and **`leaf_classifier.json`** into
`field_assistant/assets/models/`, then `flutter run --release`. The camera button appears in the chat.